# California Housing End-to-End Tabular Regression

This notebook uses the site's local CSV. Each row is a 1990 census block group, not an individual home; model selection uses validation and test is evaluated once at the end.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path('../../datasets/tabular-regression/california-housing.csv').resolve()
EXPECTED_SHA256 = '4f1001946276d792fe9935184d13296c87392aa66be32f9d26b08e623a35f0d5'
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == EXPECTED_SHA256
FEATURES = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
TARGET = 'MedHouseVal'
df = pd.read_csv(DATA_PATH)
assert df.shape == (20640, 11)
print(df.shape)
print(df[['row_id', *FEATURES, TARGET, 'split']].head(3).to_string(index=False))

(20640, 11)
  row_id  MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  Longitude  MedHouseVal      split
ca-00001  8.3252        41  6.984127   1.023810         322  2.555556     37.88    -122.23        4.526 validation
ca-00002  8.3014        21  6.238137   0.971880        2401  2.109842     37.86    -122.22        3.585 validation
ca-00003  7.2574        52  8.288136   1.073446         496  2.802260     37.85    -122.24        3.521      train


In [2]:
train = df[df['split'] == 'train'].copy()
validation = df[df['split'] == 'validation'].copy()
test = df[df['split'] == 'test'].copy()
print({'train': len(train), 'validation': len(validation), 'test': len(test)})
print('missing:', int(train.isna().sum().sum()))
print(train[[*FEATURES, TARGET]].describe().round(4).to_string())

{'train': 12384, 'validation': 4128, 'test': 4128}
missing: 0
           MedInc    HouseAge    AveRooms   AveBedrms  Population    AveOccup    Latitude   Longitude  MedHouseVal
count  12384.0000  12384.0000  12384.0000  12384.0000  12384.0000  12384.0000  12384.0000  12384.0000   12384.0000
mean       3.8602     28.6989      5.4198      1.0971   1431.6444      3.0739     35.6329   -119.5725       2.0675
std        1.8938     12.5928      2.3868      0.4788   1147.7831      7.4207      2.1300      2.0008       1.1509
min        0.4999      1.0000      1.1304      0.3333      8.0000      0.6923     32.5500   -124.3500       0.1500
25%        2.5602     18.0000      4.4447      1.0066    788.0000      2.4375     33.9300   -121.8000       1.1920
50%        3.5313     29.0000      5.2285      1.0494   1174.0000      2.8300     34.2500   -118.4900       1.7980
75%        4.7302     37.0000      6.0478      1.1000   1743.2500      3.2888     37.7100   -118.0100       2.6452
max       15.0001 

In [3]:
scaler = StandardScaler().fit(train[FEATURES])
X_train = scaler.transform(train[FEATURES])
X_validation = scaler.transform(validation[FEATURES])
print('fit partition: train')
print(pd.DataFrame({'feature': FEATURES, 'mean': scaler.mean_, 'scale': scaler.scale_}).round(6).to_string(index=False))

fit partition: train
   feature        mean       scale
    MedInc    3.860184    1.893749
  HouseAge   28.698886   12.592321
  AveRooms    5.419820    2.386676
 AveBedrms    1.097054    0.478810
Population 1431.644380 1147.736762
  AveOccup    3.073866    7.420362
  Latitude   35.632857    2.129910
 Longitude -119.572471    2.000717


In [4]:
def metrics(actual, predicted):
    return {
        'rmse': float(mean_squared_error(actual, predicted) ** 0.5),
        'mae': float(mean_absolute_error(actual, predicted)),
        'r2': float(r2_score(actual, predicted)),
    }

baseline = LinearRegression().fit(X_train, train[TARGET])
baseline_validation = metrics(validation[TARGET], baseline.predict(X_validation))
print(baseline_validation)
print(pd.Series(baseline.coef_, index=FEATURES).round(6).to_string())

{'rmse': 0.7313909507247379, 'mae': 0.5400288934706889, 'r2': 0.6002268720706481}
MedInc        0.815678
HouseAge      0.109210
AveRooms     -0.232408
AveBedrms     0.268577
Population   -0.004303
AveOccup     -0.026027
Latitude     -0.906924
Longitude    -0.883898


In [5]:
ridge_path = []
for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha, solver='svd').fit(X_train, train[TARGET])
    ridge_path.append({'alpha': alpha, **metrics(validation[TARGET], ridge.predict(X_validation))})
ridge_path = pd.DataFrame(ridge_path)
best = ridge_path.loc[ridge_path['rmse'].idxmin()]
relative_improvement = (baseline_validation['rmse'] - best['rmse']) / baseline_validation['rmse']
selected_model = 'Ridge' if relative_improvement >= 0.01 else 'LinearRegression'
print(ridge_path.round(8).to_string(index=False))
print({'best_alpha': float(best['alpha']), 'relative_improvement': float(relative_improvement), 'selected_model': selected_model})

 alpha     rmse      mae       r2
  0.01 0.731391 0.540029 0.600227
  0.10 0.731391 0.540029 0.600226
  1.00 0.731396 0.540027 0.600221
 10.00 0.731454 0.540023 0.600158
100.00 0.732715 0.540621 0.598778
{'best_alpha': 0.01, 'relative_improvement': -7.232065761393108e-08, 'selected_model': 'LinearRegression'}


In [6]:
combined = df[df['split'].isin(['train', 'validation'])]
final_scaler = StandardScaler().fit(combined[FEATURES])
if selected_model == 'Ridge':
    final_model = Ridge(alpha=float(best['alpha']), solver='svd')
else:
    final_model = LinearRegression()
final_model.fit(final_scaler.transform(combined[FEATURES]), combined[TARGET])
test_prediction = final_model.predict(final_scaler.transform(test[FEATURES]))
test_metrics = metrics(test[TARGET], test_prediction)
residuals = pd.DataFrame({
    'row_id': test['row_id'].to_numpy(),
    'actual': test[TARGET].to_numpy(),
    'prediction': test_prediction,
    'residual': test[TARGET].to_numpy() - test_prediction,
})
residuals['abs_error'] = residuals['residual'].abs()
print(test_metrics)
print(residuals.nlargest(8, 'abs_error').round(6).to_string(index=False))

{'rmse': 0.7245084534899455, 'mae': 0.5296847058600581, 'r2': 0.610047983086448}
  row_id  actual  prediction  residual  abs_error
ca-06689 5.00001    0.833245  4.166765   4.166765
ca-04493 1.12500    4.863160 -3.738160   3.738160
ca-13767 5.00001    1.369427  3.630583   3.630583
ca-10455 5.00001    1.511386  3.488624   3.488624
ca-00460 5.00001    1.537531  3.462479   3.462479
ca-04631 5.00001    1.676463  3.323547   3.323547
ca-15616 5.00001    1.762013  3.237997   3.237997
ca-08924 5.00001    1.840160  3.159850   3.159850
